# Activity 2 – Pandas
**Course:** Advanced Programming – Week 4  
**Author:** Sebastian Diaz  

Practice utilising pandas Series and DataFrame objects for storing, organising and manipulating data.

In [ ]:
import pandas as pd
import numpy as np

---
## Exercise 1 – Converting Python Data Structures to Pandas

Revisit the Week 1 data structures notebook and convert lists, dicts, and matrices into pandas Series and DataFrames.

In [ ]:
# --- List → Pandas Series ---
animals = ['elephant', 'panda', 'mouse', 'penguin', 'python']
animals_series = pd.Series(animals)
print("List → Series:")
print(animals_series)
print()

In [ ]:
# --- Dict → Pandas Series (keys become the index) ---
person = {'name': 'Frank', 'age': 56, 'height': 74}
person_series = pd.Series(person)
print("Dict → Series (keys as index):")
print(person_series)
print()

In [ ]:
# --- Dict of lists → Pandas DataFrame ---
# Cross-referencing dictionaries from the data structures notebook
departments = {"Accounts": 'Ac', "Sales": 'S', "Customer Service": 'CS'}
dept_heads  = {'Ac': 'Alison', 'S': 'Frank', 'CS': 'James'}

# Build a DataFrame combining the two
dept_df = pd.DataFrame({
    'Department': list(departments.keys()),
    'Code':       list(departments.values()),
    'Head':       [dept_heads[code] for code in departments.values()]
})
print("Dicts → DataFrame:")
print(dept_df)
print()

In [ ]:
# --- Matrix (list of lists) → Pandas DataFrame ---
matrix = [["1A", "2A", "3A", "4A"],
          ["1B", "2B", "3B", "4B"],
          ["1C", "2C", "3C", "4C"]]

matrix_df = pd.DataFrame(matrix, columns=['Col1','Col2','Col3','Col4'],
                         index=['Row1','Row2','Row3'])
print("Matrix → DataFrame:")
print(matrix_df)
print()

In [ ]:
# --- Set → Series (sets are unordered, converted via list) ---
even = {2, 4, 6, 8, 10}
even_series = pd.Series(sorted(list(even)), name='even_numbers')
print("Set → Series:")
print(even_series)

---
## Exercise 2 – Sales Data Analysis

File: `Activity_2_Exercise_2_SalesData.csv`  
Stores: L3, L1, P2, N6, N4, B8 | Months: Apr-18 to Mar-19

In [ ]:
# Load data
df = pd.read_csv('Activity_2_Exercise_2_SalesData.csv', index_col=0)
print("Sales Data:")
print(df)
print("\nShape:", df.shape)

In [ ]:
# --- Q1: Sales for P2 and B8 in Nov-18, Feb-19 and Mar-19 ---
stores_q1 = ['P2', 'B8']
months_q1 = ['Nov-18', 'Feb-19', 'Mar-19']

result_q1 = df.loc[stores_q1, months_q1]
print("Q1 – P2 and B8 in Nov-18, Feb-19, Mar-19:")
print(result_q1)

In [ ]:
# --- Q2: Q3 (Oct-18 to Dec-18) for London stores L3 and L1, monthly % increase ---
q3_months = ['Oct-18', 'Nov-18', 'Dec-18']
london_q3 = df.loc[['L3', 'L1'], q3_months]
print("Q2 – London Stores in Q3 (Oct-18 to Dec-18):")
print(london_q3)
print()

# Monthly percentage increase (relative to previous month)
pct_change = london_q3.pct_change(axis=1) * 100
print("Monthly % increase (Oct→Nov, Nov→Dec):")
print(pct_change.round(2))

In [ ]:
# --- Q3: Top three months for New York stores (N6 and N4) ---
ny_df = df.loc[['N6', 'N4'], :]

# Option A: Top 3 months by combined (N6 + N4) sales
ny_combined = ny_df.sum(axis=0)
top3_combined = ny_combined.nlargest(3)
print("Q3 – Top 3 months for NY (N6 + N4 combined):")
for month, total in top3_combined.items():
    print(f"  {month}: combined total = {total:.0f} (N6={ny_df.loc['N6', month]}, N4={ny_df.loc['N4', month]})")
print()

# Option B: Top 3 individual store-month combinations
ny_stack = ny_df.stack().reset_index()
ny_stack.columns = ['Store', 'Month', 'Sales']
top3_individual = ny_stack.nlargest(3, 'Sales')
print("Q3 – Top 3 individual store-month results:")
print(top3_individual.to_string(index=False))

In [ ]:
# --- Q4: Overall lowest sales figure, store and month ---
min_value = df.values.min()
min_loc = np.argwhere(df.values == min_value)
min_store = df.index[min_loc[0][0]]
min_month = df.columns[min_loc[0][1]]

print(f"Q4 – Overall lowest sales: {min_value}")
print(f"     Store : {min_store}")
print(f"     Month : {min_month}")

# Pandas idxmin approach
flat_min = df.stack().idxmin()
print(f"\nVerification via .stack().idxmin(): Store={flat_min[0]}, Month={flat_min[1]}")

---
## Exercise 3 – Percentage Increase vs Annual Average

In [ ]:
# Calculate each store's annual average
annual_avg = df.mean(axis=1)   # mean across all 12 months for each store

# % increase for each month relative to the store's annual average
# pct_increase[store, month] = ((month_value - annual_avg) / annual_avg) * 100
pct_increase = df.subtract(annual_avg, axis=0).divide(annual_avg, axis=0) * 100
pct_increase = pct_increase.round(2)

print("Percentage increase vs annual average (each store):")
print(pct_increase)
print()
print("Annual averages per store:")
print(annual_avg.round(2))

In [ ]:
# --- Q3 Ex3 Q1: Store with largest and smallest increase, and in which month ---
# Largest increase across all stores
max_pct = pct_increase.stack().idxmax()
min_pct = pct_increase.stack().idxmin()

print(f"Largest increase  : Store {max_pct[0]}, Month {max_pct[1]} "
      f"({pct_increase.loc[max_pct[0], max_pct[1]]:.2f}%)")
print(f"Smallest increase : Store {min_pct[0]}, Month {min_pct[1]} "
      f"({pct_increase.loc[min_pct[0], min_pct[1]]:.2f}%)")

In [ ]:
# --- Q3 Ex3 Q2: Months with smallest and largest average % increase across all stores ---
monthly_avg_pct = pct_increase.mean(axis=0)   # average across stores per month

print("Average % increase per month (across all stores):")
print(monthly_avg_pct.round(2))
print()
print(f"Month with largest  average increase : {monthly_avg_pct.idxmax()} "
      f"({monthly_avg_pct.max():.2f}%)")
print(f"Month with smallest average increase : {monthly_avg_pct.idxmin()} "
      f"({monthly_avg_pct.min():.2f}%)")

---
## Exercise 4 – Discussion: NumPy ndarrays vs Pandas (approx. 200 words)

Having used the same sales dataset in both NumPy (Activity 1) and Pandas (Activity 2), the differences in approach are clear.

**NumPy** required manual management of row and column headers as separate arrays. Accessing data by store name (e.g. `N6`) meant finding its index first with `np.where()`, then using integer indexing. This is verbose but very fast and memory-efficient, making NumPy ideal for large-scale purely numerical computations.

**Pandas**, by contrast, made the same tasks significantly more readable. Using `df.loc['N6', 'Nov-18']` is far more intuitive than `data[n6_idx, 7]`. Functions like `pct_change()`, `idxmin()`, `nlargest()`, and `stack()` removed the need to write custom logic for common analytical patterns. The DataFrame's native support for heterogeneous data types and NaN handling also means it is better suited to real-world datasets where missing values are expected.

For a **simple numerical dataset** like the sales data, Pandas is the better choice due to readability and the rich built-in API. NumPy would be preferable for **large homogeneous matrices** (e.g. image processing, simulations, linear algebra) where raw computational speed is the priority and labels are not needed.

In future, I would default to **Pandas for data analysis tasks** and reserve **NumPy for mathematical computation and when interfacing with ML libraries** like TensorFlow or scikit-learn, which often require plain ndarrays.